In [8]:
!pip install xgboost -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor

from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings("ignore")# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/sayankrdutta/amazon-dataset/Amazon.csv


In [9]:
DATA_PATH = "/kaggle/input/datasets/sayankrdutta/amazon-dataset/Amazon.csv"

df = pd.read_csv(DATA_PATH)

print(df.shape)
df.head()

(100000, 20)


,OrderID,OrderDate,CustomerID,CustomerName,ProductID,ProductName,Category,Brand,Quantity,UnitPrice,Discount,Tax,ShippingCost,TotalAmount,PaymentMethod,OrderStatus,City,State,Country,SellerID
0,ORD0000001,2023-01-31,CUST001504,Vihaan Sharma,P00014,Drone Mini,Books,BrightLux,3,106.59,0.00,0.00,0.09,319.86,Debit Card,Delivered,Washington,DC,India,SELL01967
1,ORD0000002,2023-12-30,CUST000178,Pooja Kumar,P00040,Microphone,Home & Kitchen,UrbanStyle,1,251.37,0.05,19.10,1.74,259.64,Amazon Pay,Delivered,Fort Worth,TX,United States,SELL01298
2,ORD0000003,2022-05-10,CUST047516,Sneha Singh,P00044,Power Bank 20000mAh,Clothing,UrbanStyle,3,35.03,0.10,7.57,5.91,108.06,Debit Card,Delivered,Austin,TX,United States,SELL00908
3,ORD0000004,2023-07-18,CUST030059,Vihaan Reddy,P00041,Webcam Full HD,Home & Kitchen,Zenith,5,33.58,0.15,11.42,5.53,159.66,Cash on Delivery,Delivered,Charlotte,NC,India,SELL01164
4,ORD0000005,2023-02-04,CUST048677,Aditya Kapoor,P00029,T-Shirt,Clothing,KiddoFun,2,515.64,0.25,38.67,9.23,821.36,Credit Card,Cancelled,San Antonio,TX,Canada,SELL01411


In [10]:
# Standardize
df.columns = df.columns.str.lower()

# Convert date
df["orderdate"] = pd.to_datetime(df["orderdate"])

# Extract features from date
df["year"] = df["orderdate"].dt.year
df["month"] = df["orderdate"].dt.month
df["day"] = df["orderdate"].dt.day

# Drop non-useful identifiers
df.drop([
    "orderid", "customerid", "sellerid", "productid",
    "customername", "productname"
], axis=1, inplace=True)

# Encode categorical variables
df = pd.get_dummies(df, drop_first=True)

# Handle missing
df.fillna(df.median(), inplace=True)

In [14]:
target = "totalamount"

X = df.drop(target, axis=1)
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [15]:
X_train = X_train.select_dtypes(include=[np.number])
X_test = X_test.select_dtypes(include=[np.number])

models = {
    "Linear": LinearRegression(),
    "DecisionTree": DecisionTreeRegressor(random_state=42),
    "KNN": KNeighborsRegressor()
}

rq1 = []

for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    rq1.append([
        name,
        mean_absolute_error(y_test, pred),
        np.sqrt(mean_squared_error(y_test, pred)),
        r2_score(y_test, pred)
    ])

rq1_df = pd.DataFrame(rq1, columns=["Model", "MAE", "RMSE", "R2"])
rq1_df.to_csv("RQ1_table.csv", index=False)

plt.figure()
plt.bar(rq1_df["Model"], rq1_df["R2"])
plt.savefig("RQ1_figure.pdf")
plt.close()

In [33]:
# ===== RQ2: Model Comparison =====

models = {
    "RandomForest": RandomForestRegressor(
        n_estimators=20,
        max_depth=10,
        n_jobs=-1,
        random_state=42
    ),
    "XGBoost": XGBRegressor(
        n_estimators=50,
        max_depth=6,
        learning_rate=0.1,
        random_state=42
    ),
    "SVM": SVR()
}

rq2 = []

for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    rq2.append([
        name,
        mean_absolute_error(y_test, pred),
        np.sqrt(mean_squared_error(y_test, pred)),
        r2_score(y_test, pred)
    ])

rq2_df = pd.DataFrame(rq2, columns=["Model", "MAE", "RMSE", "R2"])

rq2_df.to_csv("RQ2_table.csv", index=False)

plt.figure()
plt.bar(rq2_df["Model"], rq2_df["R2"])
plt.title("RQ2 Model Comparison")
plt.savefig("RQ2_figure.pdf")
plt.close()

In [19]:
from sklearn.preprocessing import StandardScaler

# Ensure only numeric features are used
X_numeric = X.select_dtypes(include=[np.number])

# Train-test split for raw data
X_train, X_test, y_train, y_test = train_test_split(
    X_numeric, y, test_size=0.2, random_state=42
)

# Apply scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_numeric)

# Train-test split for scaled data
X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

# Model
model = XGBRegressor(random_state=42)

# Raw model
model.fit(X_train, y_train)
pred_raw = model.predict(X_test)

# Scaled model
model.fit(X_train_s, y_train_s)
pred_scaled = model.predict(X_test_s)

# Results table
rq3_df = pd.DataFrame([
    ["Raw",
     mean_absolute_error(y_test, pred_raw),
     np.sqrt(mean_squared_error(y_test, pred_raw)),
     r2_score(y_test, pred_raw)],
    
    ["Scaled",
     mean_absolute_error(y_test_s, pred_scaled),
     np.sqrt(mean_squared_error(y_test_s, pred_scaled)),
     r2_score(y_test_s, pred_scaled)]
], columns=["Preprocessing", "MAE", "RMSE", "R2"])

rq3_df.to_csv("RQ3_table.csv", index=False)

# Plot
plt.figure()
plt.plot(rq3_df["Preprocessing"], rq3_df["R2"], marker='o')
plt.title("RQ3 Preprocessing Impact")
plt.savefig("RQ3_figure.pdf")
plt.close()

In [22]:
# Ensure same features used for training
X_numeric = X.select_dtypes(include=[np.number])

X_train, X_test, y_train, y_test = train_test_split(
    X_numeric, y, test_size=0.2, random_state=42
)

model = XGBRegressor(random_state=42)
model.fit(X_train, y_train)

# FIX: use X_train.columns (not X.columns)
rq4_df = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": model.feature_importances_
}).sort_values(by="Importance", ascending=False)

rq4_df.to_csv("RQ4_table.csv", index=False)

plt.figure()
plt.barh(rq4_df["Feature"][:10], rq4_df["Importance"][:10])
plt.title("Top 10 Feature Importance")
plt.savefig("RQ4_figure.pdf")
plt.close()

In [24]:
rq5_df = rq2_df.copy()

rq5_df["MAE_rank"] = rq5_df["MAE"].rank()
rq5_df["RMSE_rank"] = rq5_df["RMSE"].rank()
rq5_df["R2_rank"] = rq5_df["R2"].rank(ascending=False)

rq5_df.to_csv("RQ5_table.csv", index=False)

plt.figure()
plt.plot(rq5_df["Model"], rq5_df["R2_rank"], marker='o')
plt.savefig("RQ5_figure.pdf")
plt.close()

In [30]:
# ===== RQ6: Robustness Check =====

# STEP 1: Clean X (IMPORTANT)
X_clean = X.copy()

if "orderdate" in X_clean.columns:
    X_clean = X_clean.drop("orderdate", axis=1)

# keep only numeric columns
X_clean = X_clean.select_dtypes(include=[np.number])

# STEP 2: Model
model = XGBRegressor(random_state=42)

# STEP 3: Cross-validation
cv_scores = cross_val_score(model, X_clean, y, cv=5, scoring="r2")

# STEP 4: Add noise
X_noise = X_clean + np.random.normal(0, 0.1, X_clean.shape)

# STEP 5: Train on noisy data
model.fit(X_noise, y)
pred_noise = model.predict(X_noise)

# STEP 6: Results table
rq6_df = pd.DataFrame({
    "Scenario": ["CV", "Noise"],
    "R2": [cv_scores.mean(), r2_score(y, pred_noise)]
})

# STEP 7: Save results
rq6_df.to_csv("RQ6_table.csv", index=False)

# STEP 8: Plot
plt.figure()
plt.plot(rq6_df["Scenario"], rq6_df["R2"], marker='o')
plt.title("RQ6 Robustness")
plt.savefig("RQ6_figure.pdf")
plt.close()

In [32]:
rq7_df = rq2_df.copy()

rq7_df["Score"] = rq7_df["R2"] - 0.01*rq7_df["RMSE"] - 0.01*rq7_df["MAE"]

rq7_df.to_csv("RQ7_table.csv", index=False)

plt.figure()
plt.bar(rq7_df["Model"], rq7_df["Score"])
plt.savefig("RQ7_figure.pdf")
plt.close()